# Module 5: KRaft Architecture

本章目标：
- 理解 ZooKeeper 模式的局限性及为何被替换
- 掌握 KRaft（Kafka Raft Metadata）的核心架构
- 通过 Admin API 查询 KRaft 集群元数据
- 解读本教程的 `docker-compose.yml` 中 KRaft 配置

---

## 历史背景：ZooKeeper → KRaft

### ZooKeeper 时代（Kafka < 2.8）

```mermaid
graph TD
    subgraph zk["ZooKeeper Ensemble"]
        Z1["zk-1"] <--> Z2["zk-2"] <--> Z3["zk-3"]
        Z1 <--> Z3
    end
    subgraph kafka["Kafka Cluster"]
        B1["broker-1"]
        B2["broker-2"]
        B3["broker-3"]
    end
    zk <-->|"元数据 （独立运维）"| kafka
```

**痛点：**
- 需要单独运维 ZooKeeper 集群（额外复杂度）
- ZooKeeper 存储元数据有上限（Broker 数、Topic 数受限）
- 控制器（Controller）切换时需重新从 ZooKeeper 加载大量状态，影响可用性

### KRaft 时代（Kafka 3.x+，4.0 默认）

```mermaid
graph TD
    subgraph kraft["KRaft Kafka Cluster（无 ZooKeeper）"]
        K1["kafka-1 broker + controller"]
        K2["kafka-2 broker + controller"]
        K3["kafka-3 broker + controller"]
        K1 <-->|"Raft 共识"| K2
        K2 <-->|"Raft 共识"| K3
        K1 <-->|"Raft 共识"| K3
    end
    AC["Active Controller （由 Raft 选举产生）"]
    K1 -.->|"当选"| AC
```

**优势：**
- 无外部依赖，运维简化
- 支持百万级 Partition（元数据存储于 Kafka 内部 topic）
- 控制器故障切换更快（本地状态，无需重新加载）

## 解读 docker-compose.yml 中的 KRaft 配置

```yaml
KAFKA_NODE_ID: 1                  # Broker/Controller 唯一 ID
KAFKA_PROCESS_ROLES: broker,controller  # 同时担任 Broker 和 Controller
KAFKA_CONTROLLER_QUORUM_VOTERS: 1@kafka-1:9093,2@kafka-2:9093,3@kafka-3:9093
                                  # Raft 投票成员列表：node_id@host:port
KAFKA_LISTENERS: PLAINTEXT://0.0.0.0:9092,EXTERNAL://0.0.0.0:9094,CONTROLLER://0.0.0.0:9093
KAFKA_ADVERTISED_LISTENERS: PLAINTEXT://kafka-1:9092,EXTERNAL://localhost:19094
KAFKA_CONTROLLER_LISTENER_NAMES: CONTROLLER  # Controller 间通信使用此 listener
KAFKA_CLUSTER_ID: abcdefghijklmnopqrstuv     # 22字符 Base64 UUID，集群唯一标识
```

### PROCESS_ROLES 选项

| 配置 | 说明 | 适用场景 |
|------|------|----------|
| `broker,controller` | 同时是 Broker 和 Controller 候选 | 小集群（本教程用法）|
| `broker` | 纯 Broker，不参与 Controller 选举 | 大集群的 Broker 节点 |
| `controller` | 纯 Controller，不处理客户端请求 | 大集群的专用控制节点 |

In [ ]:
import asyncio
from aiokafka.admin import AIOKafkaAdminClient

BROKERS = "localhost:19094,localhost:29094,localhost:39094"

## 1. 查询集群元数据

In [ ]:
async def describe_cluster(brokers):
    admin = AIOKafkaAdminClient(bootstrap_servers=brokers)
    await admin.start()
    try:
        cluster = await admin.describe_cluster()
        print(f"Cluster ID: {cluster['cluster_id']}")
        print(f"Active Controller: node_id={cluster['controller_id']}")
        print(f"\nBrokers ({len(cluster['brokers'])})：")
        for b in cluster['brokers']:
            print(f"  node_id={b['node_id']}, host={b['host']}, port={b['port']}")
    finally:
        await admin.close()

await describe_cluster(BROKERS)

## 2. 查询所有 Topic 及其分区分布

In [ ]:
async def list_all_topics(brokers):
    admin = AIOKafkaAdminClient(bootstrap_servers=brokers)
    await admin.start()
    try:
        topics = await admin.list_topics()
        # 过滤内部 topic
        user_topics = [t for t in sorted(topics) if not t.startswith('_')]
        print(f"User topics ({len(user_topics)}):")
        for t in user_topics:
            print(f"  {t}")

        if user_topics:
            print(f"\nPartition details:")
            descriptions = await admin.describe_topics(user_topics)
            for desc in descriptions:
                partitions = desc['partitions']
                print(f"  {desc['topic']}: {len(partitions)} partition(s)")
    finally:
        await admin.close()

await list_all_topics(BROKERS)

## 3. Raft 选举过程（概念演示）

KRaft 使用 Raft 共识算法选出 Active Controller。以下模拟说明选举过程：

```
初始状态：所有节点都是 Follower

1. 选举超时触发：node-1 成为 Candidate
   node-1: "请投票给我！（term=1）"
   node-2: "同意" → node-1 获得多数票 → 成为 Leader (Active Controller)
   node-3: "同意"

2. 正常运行：
   Leader node-1 定期发送心跳，Follower 重置选举计时器
   所有元数据变更都先写入 Leader，再复制到 Follower

3. Leader 故障：
   node-2 和 node-3 超时后发起新选举（term=2）
   其中一个获得多数票成为新 Controller
   → 整个过程通常在几秒内完成
```

In [ ]:
async def watch_controller(brokers, rounds=8, interval=3):
    """监测 Active Controller 的变化。"""
    admin = AIOKafkaAdminClient(bootstrap_servers=brokers)
    await admin.start()
    try:
        prev_controller = None
        for r in range(rounds):
            try:
                cluster = await admin.describe_cluster()
                ctrl = cluster['controller_id']
                if ctrl != prev_controller:
                    if prev_controller is not None:
                        print(f"  Controller changed: {prev_controller} -> {ctrl}")
                    else:
                        print(f"  Active Controller: node_id={ctrl}")
                    prev_controller = ctrl
                else:
                    print(f"  Round {r+1}: Controller = {ctrl} (no change)")
            except Exception as e:
                print(f"  Round {r+1}: Error - {e}")
            await asyncio.sleep(interval)
    finally:
        await admin.close()

print("监测 Controller（可在另一个终端 stop/start 某个 kafka broker）")
print("24 秒后自动停止\n")
await watch_controller(BROKERS)

## 总结

| 项目 | ZooKeeper 模式 | KRaft 模式 |
|------|--------------|----------|
| 外部依赖 | 需要 ZooKeeper 集群 | 无 |
| 元数据存储 | ZooKeeper znodes | Kafka 内部 __cluster_metadata topic |
| 控制器切换 | 秒级（需加载大量状态）| 毫秒级（本地日志）|
| 最大 Partition 数 | ~200,000 | 数百万 |
| 生产可用 | Kafka < 3.x | Kafka 3.3+（KIP-833），4.0 唯一模式 |

## 下一章

**Module 6: Advanced Producer** — 消息压缩、自定义 Partitioner、幂等与事务生产者。